# UCF-Crime on AnomalyCLIP CLIP features — Binary (Normal vs Anomaly) → 13-class anomaly type

Trains and tunes classifiers on the pre-extracted **ViT-B/16 CLIP features** of the AnomalyCLIP release (`Image-Features/`, `Annotations/`, `Temporal_Anomaly_Annotation_for_Testing_Videos.txt`).

**Before the first run**
1. Add your dataset to the notebook (*Add Input*), enable **GPU** and **Internet** (only needed if `optuna` is missing).
2. Set `FAST_DEBUG = True` in the config cell and run everything once (~5-10 min). If it finishes, set it back to `False` for the real run.
3. Read the printouts of Steps 2-4 — they show what was found (file shapes, class counts, official split). The notebook cannot see your data layout in advance, so it *discovers* it; if the counts are not **train 1,610 / test 290** (810+800 / 140+150), fix the split before trusting any metric.

**Design (what and why)**

| Problem | Decision |
|---|---|
| The features are ~28 GB (one 512-d vector per frame) — impossible to hold in Kaggle RAM | Stream each file once with memory-mapping and mean-pool it into `N_SEGMENTS` (64) equal temporal segments → ~0.25 GB, cached per video (resumable) |
| Train labels are **video-level only** (weak); frame-level truth exists only for test videos | Train with video-level labels (MIL: a video's score = mean of its top-k segment scores); use the test temporal annotations only for *scoring* frame-level AUC |
| Only 1,610 training videos → a single validation split is noisy and overfits easily | Hyper-parameters tuned with **stratified K-fold CV** (Optuna TPE + pruning); final models = cross-validated multi-seed ensembles |
| Threshold / model choice must not touch the test set | Both are chosen from **out-of-fold** predictions. The official test set is evaluated once |
| Normal vs anomaly in CLIP space | Models: LogReg, LightGBM (video descriptors: mean/max/std over time) and a temporal neural net (transformer / BiGRU / conv). The net can *re-center* features on the **normal prototype** (the central idea of AnomalyCLIP), tuned as a hyper-parameter |
| 13-way task | Only anomalous videos; gated-attention pooling over segments; class weighting & label smoothing tuned |

**Tuned (Optuna):** optimizer (AdamW / RAdam / SGD-Nesterov), learning rate, weight decay, β2, batch size, epochs (cosine schedule with warm-up), architecture / width / depth / heads, dropout, input dropout, feature noise, temporal masking, label smoothing, top-k fraction, sparsity & smoothness weights, class weights, re-centering.

**Comparable to published work** (the final comparison table checks the split first and refuses to call the run comparable otherwise):

| Paper | What this notebook reproduces | Caveat |
|---|---|---|
| Sultani et al., CVPR 2018 | frame-level AUC on the official 290 test videos; false-alarm rate on normal test videos at 0.5; a *Sultani-style* model (32 segments, 512-32-1 FC net, MIL ranking loss, Adagrad 1e-3, λ=8e-5, 30+30 bags) | the paper used C3D features; here the method runs on CLIP features |
| AnomalyCLIP, CVIU 2024 | frame-level AUC; frame-level one-vs-rest **mAUC** over the 13 classes | frame class scores are *approximated* (anomaly score × video class probability); the paper predicts per-frame class probabilities |

**Outputs** (`/kaggle/working/artifacts`): `results.json`, `test_predictions.csv`, model ensembles (`nn_models.pt`, `classical_models.joblib`), result tables. The pooled-feature cache folder can be saved as a dataset to skip the slow pass next time.

## Step 1 — Setup & configuration

In [ ]:
import os, re, sys, json, math, time, random, hashlib, warnings, importlib, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

for _pkg in ["optuna", "lightgbm"]:                      # Kaggle images usually have both; install only if missing
    try:
        importlib.import_module(_pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)

import optuna, lightgbm as lgb
import torch, torch.nn as nn, torch.nn.functional as F
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, accuracy_score, precision_score,
                             recall_score, matthews_corrcoef, confusion_matrix, balanced_accuracy_score,
                             log_loss, brier_score_loss, classification_report, roc_curve, precision_recall_curve)
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kw): return x
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =====================  CONFIG  =====================
SEED = 42
FAST_DEBUG = False        # True -> ~5 minute smoke test (tiny search, few epochs). Do this once before the full run.

N_SEGMENTS       = 64     # every video is mean-pooled into this many equal temporal segments
NORMALIZE_FRAMES = True   # L2-normalise every frame feature before pooling (CLIP convention)
FRAME_STRIDE     = 1      # original video frames per feature row (1 = features exist for every frame)
MAX_VIDEOS       = None   # e.g. 120 to debug the data pipeline on a subset
IO_THREADS       = 4

N_FOLDS_TUNE  = 3         # CV folds inside the Optuna search (speed)
N_FOLDS_FINAL = 5         # CV folds for the final models / out-of-fold predictions
N_SEEDS_FINAL = 3         # seeds per fold for the final neural ensemble
MAX_EPOCHS    = 60
PATIENCE      = 10

N_TRIALS    = {"bin_nn": 40, "bin_lgbm": 25, "bin_lr": 20, "mc_nn": 40, "mc_lgbm": 20, "mc_lr": 20}
TIMEOUT_MIN = {"bin_nn": 25, "bin_lgbm": 10, "bin_lr": 5,  "mc_nn": 25, "mc_lgbm": 10, "mc_lr": 5}

if FAST_DEBUG:
    N_FOLDS_TUNE, N_FOLDS_FINAL, N_SEEDS_FINAL, MAX_EPOCHS, PATIENCE = 2, 3, 1, 6, 4
    N_TRIALS    = {k: 3 for k in N_TRIALS}
    TIMEOUT_MIN = {k: 3 for k in TIMEOUT_MIN}

CATS = ["Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion", "Fighting",
        "RoadAccidents", "Robbery", "Shooting", "Shoplifting", "Stealing", "Vandalism"]   # the 13 anomaly classes
BIN_NAMES = ["Normal", "Anomaly"]
INPUT_ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
OUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUT / f"seg_cache_N{N_SEGMENTS}_n{int(NORMALIZE_FRAMES)}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_everything(SEED)
print("device:", DEVICE, "| torch", torch.__version__, "| optuna", optuna.__version__, "| lightgbm", lgb.__version__)
print("FAST_DEBUG:", FAST_DEBUG, "| segments per video:", N_SEGMENTS, "| output dir:", OUT)


## Step 2 — Discover files (feature files, annotation files, temporal annotation file)

In [ ]:
# ---- one pass over /kaggle/input: find feature files, annotation files, temporal-annotation file ----
FEATURE_SUFFIXES = {".npy", ".npz", ".pt", ".pth"}
all_feature_files, annotation_files, temporal_files = [], [], []
n_seen = 0
SCAN_ROOTS = [INPUT_ROOT] + ([EXTRACT_ROOT] if globals().get("EXTRACT_ROOT") else [])   # EXTRACT_ROOT is set by the unzip cell
def shown(p):
    for r in SCAN_ROOTS:
        try: return str(Path(p).relative_to(r))
        except ValueError: pass
    return str(p)
for dp, dns, fns in (x for r in SCAN_ROOTS for x in os.walk(r)):
    for f in fns:
        n_seen += 1
        p = Path(dp) / f
        if "seg_cache_" in str(p): continue                      # pooled cache from a previous run, not raw features
        suf, low = p.suffix.lower(), f.lower()
        low_parts = [x.lower() for x in p.parts]
        if suf in FEATURE_SUFFIXES:
            all_feature_files.append(p)
        elif "temporal" in low and suf in {".txt", ".csv"}:
            temporal_files.append(p)
        elif suf in {".txt", ".csv", ".json"} and any("annotation" in x for x in low_parts):
            annotation_files.append(p)
    if n_seen > 400_000:
        raise RuntimeError("More than 400k files under /kaggle/input. This notebook expects ONE feature file per video "
                           "(not one per frame). Check the dataset layout.")

FEATURE_DIR_RE = re.compile(r"image[-_ ]?features?", re.I)
in_feat_dir = [p for p in all_feature_files if any(FEATURE_DIR_RE.fullmatch(x) for x in p.parts)]
feature_files = in_feat_dir if in_feat_dir else all_feature_files
assert feature_files, "No .npy/.npz/.pt feature files found under /kaggle/input. Attach your dataset to the notebook."

if in_feat_dir:
    p0 = in_feat_dir[0]
    idx = [i for i, x in enumerate(p0.parts) if FEATURE_DIR_RE.fullmatch(x)][0]
    FEATURE_ROOT = Path(*p0.parts[: idx + 1])
else:
    FEATURE_ROOT = Path(os.path.commonpath([str(p.parent) for p in feature_files]))

print("Feature root       :", FEATURE_ROOT)
print("Feature files      :", len(feature_files), pd.Series([p.suffix.lower() for p in feature_files]).value_counts().to_dict())
print("Annotation files   :", len(annotation_files))
for p in annotation_files[:15]: print("   ", shown(p))
print("Temporal annot.    :", [shown(p) for p in temporal_files])
print("\nFirst feature files:")
for p in sorted(feature_files)[:5]: print("   ", p.relative_to(FEATURE_ROOT))

# ---- look at real array shapes before building anything ----
def load_raw(path):
    path = Path(path)
    if path.suffix.lower() == ".npy":
        return np.load(path, mmap_mode="r")
    if path.suffix.lower() == ".npz":
        z = np.load(path); k = "features" if "features" in z.files else z.files[0]; return z[k]
    obj = torch.load(path, map_location="cpu")
    return obj.numpy() if hasattr(obj, "numpy") else np.asarray(obj)

print("\nShape / dtype of a few feature files:")
for p in sorted(feature_files)[:: max(1, len(feature_files) // 6)][:6]:
    a = load_raw(p); print(f"   {p.name:40s} shape={tuple(a.shape)} dtype={a.dtype}")


## Step 3 — One row per video; class is read from the file name (`Abuse001_x264` → Abuse, `Normal_Videos_…` → Normal)

In [ ]:
# ---- build ONE row per video (files of the same video, e.g. several crops, are grouped) ----
CAT_LOOKUP = {re.sub(r"[^a-z]", "", c.lower()): c for c in CATS}

def video_stem(raw_stem):
    # 'Abuse001_x264__3' -> 'Abuse001_x264'  (drop crop / suffix after 'x264')
    return raw_stem[: raw_stem.index("x264") + 4] if "x264" in raw_stem else re.sub(r"(__|_)\d+$", "", raw_stem)

def category_of(stem, rel_parts):
    s = stem.lower()
    if s.startswith("normal"):
        return "Normal"
    m = re.match(r"^([a-z_]+?)_?\d", s)
    if m and re.sub(r"[^a-z]", "", m.group(1)) in CAT_LOOKUP:
        return CAT_LOOKUP[re.sub(r"[^a-z]", "", m.group(1))]
    for part in rel_parts:
        k = re.sub(r"[^a-z]", "", part.lower())
        if k in CAT_LOOKUP: return CAT_LOOKUP[k]
        if k.startswith("normal"): return "Normal"
    return None

groups = {}
for p in feature_files:
    rel = p.relative_to(FEATURE_ROOT)
    stem = video_stem(p.stem)
    key = f"{rel.parent.as_posix()}/{stem}"
    g = groups.setdefault(key, {"key": key, "stem": stem, "rel_dir": rel.parent.as_posix(), "files": []})
    g["files"].append(p)

vids = pd.DataFrame(list(groups.values())).sort_values("key").reset_index(drop=True)
vids["category"] = [category_of(r.stem, Path(r.rel_dir).parts) for r in vids.itertuples()]
dropped = vids[vids.category.isna()]
if len(dropped):
    print(f"[WARNING] {len(dropped)} videos have no recognisable class and are dropped, e.g.:", dropped.key.head(5).tolist())
vids = vids[vids.category.notna()].reset_index(drop=True)
vids["label_bin"] = (vids.category != "Normal").astype(int)
vids["n_files"] = vids.files.apply(len)
if MAX_VIDEOS:
    vids = vids.groupby("label_bin", group_keys=False).apply(lambda d: d.sample(min(len(d), MAX_VIDEOS // 2), random_state=SEED)).reset_index(drop=True)

print(f"Videos: {len(vids)} | files per video: {vids.n_files.value_counts().to_dict()}")
print(vids.category.value_counts().to_string())
missing = [c for c in CATS if c not in set(vids.category)]
if missing: print("[WARNING] anomaly classes with NO videos:", missing)
dup = vids[vids.duplicated("stem", keep=False)]
if len(dup): print(f"Note: {dup.stem.nunique()} video names occur in more than one folder (kept separate, keyed by folder).")


## Step 4 — Official train / test split

**Check this output carefully** — it should read train 1,610 (800 Normal + 810 anomaly) and test 290 (150 Normal + 140 anomaly).

In [ ]:
# ---- official train / test split ----
# Priority: (1) annotation files whose NAME contains 'train' / 'test' (their lists of video names),
#           (2) the temporal-annotation file (= anomalous test videos),
#           (3) folder names containing 'train' / 'test'.
def stems_in_file(p):
    try: txt = Path(p).read_text(errors="ignore")
    except Exception: return set()
    return {s.lower() for s in re.findall(r"[A-Za-z0-9_\-]+x264", txt)}

train_stems, test_stems, ignored_files = set(), set(), []
for p in annotation_files:
    name, st = p.name.lower(), stems_in_file(p)
    if not st: ignored_files.append(p.name); continue
    if "test" in name:                      test_stems |= st
    elif "train" in name or "val" in name:  train_stems |= st
    else:                                   ignored_files.append(p.name)
temporal_stems = set()
for p in temporal_files: temporal_stems |= stems_in_file(p)
test_stems |= temporal_stems
print(f"stems from annotation files -> train: {len(train_stems)}  test: {len(test_stems)}  (of which temporal file: {len(temporal_stems)})")
if ignored_files: print("annotation files not used for splitting:", ignored_files[:10])

def path_split(rel_dir):
    low = rel_dir.lower()
    if "test" in low: return "test"
    if "train" in low: return "train"
    return None

splits = []
for r in vids.itertuples():
    s = r.stem.lower(); in_te, in_tr = s in test_stems, s in train_stems
    if in_te and not in_tr: sp = "test"
    elif in_tr and not in_te: sp = "train"
    else: sp = path_split(r.rel_dir)                # ambiguous or unknown -> folder name
    splits.append(sp)
vids["split"] = splits

# --- repairs, each announced loudly ---
if (vids.split == "test").sum() == 0:
    print("\n[WARNING] No official test videos could be identified -> using a stratified 85/15 hold-out split instead (NOT the official split).")
    idx_tr, idx_te = train_test_split(np.arange(len(vids)), test_size=0.15, random_state=SEED, stratify=vids.category)
    vids.loc[idx_te, "split"] = "test"; vids.loc[idx_tr, "split"] = "train"
unk = vids.split.isna()
if unk.any():
    print(f"[WARNING] {int(unk.sum())} videos had no split info -> assigned to TRAIN.")
    vids.loc[unk, "split"] = "train"
te_norm = int(((vids.split == "test") & (vids.label_bin == 0)).sum()); te_n = int((vids.split == "test").sum())
tr_norm = int(((vids.split == "train") & (vids.label_bin == 0)).sum())
if te_norm < 0.1 * te_n and tr_norm > 100:
    k = min(150, tr_norm // 5)
    print(f"[WARNING] the test split contains only {te_norm} Normal videos. Moving {k} random Normal videos from train to test "
          f"so Normal-vs-Anomaly can be evaluated (this deviates from the official split).")
    mv = vids[(vids.split == "train") & (vids.label_bin == 0)].sample(k, random_state=SEED).index
    vids.loc[mv, "split"] = "test"

print("\nVideos per split / class:")
print(pd.crosstab(vids.category, vids.split).to_string())
print("\n(Official UCF-Crime: train 1,610 = 810 anomaly + 800 normal | test 290 = 140 anomaly + 150 normal)")


## Step 5 — Pool per-frame features into 64 segments per video (streamed, cached, resumable)

This is the only slow I/O step (reads the whole ~28 GB once). Safe to interrupt and re-run.

In [ ]:
# ---- per-frame features (T x 512, ~28 GB in total) -> N_SEGMENTS mean-pooled segments per video (~0.25 GB) ----
# Streams every file with memory-mapping, so RAM stays small. One cache file per video => safe to interrupt and re-run.
EXPECTED_D = None

def to_td_views(arr, d_hint):
    """Return a list of [T, D] array views (one per crop if the file stores several crops)."""
    a = arr
    if a.ndim == 2:
        if d_hint and a.shape[1] != d_hint and a.shape[0] == d_hint: a = a.T
        return [a]
    if a.ndim == 3:
        f = [i for i, s in enumerate(a.shape) if s == d_hint]
        f = f[-1] if f else a.ndim - 1
        rest = [i for i in range(3) if i != f]
        c = min(rest, key=lambda i: a.shape[i]); t = [i for i in rest if i != c][0]
        a = np.transpose(a, (c, t, f))
        return [a[i] for i in range(a.shape[0])]
    raise ValueError(f"Unsupported feature array shape {a.shape}")

def pool_td(a, n_seg, normalize):
    T, D = a.shape
    out = np.empty((n_seg, D), np.float32)
    b = np.linspace(0, T, n_seg + 1).astype(np.int64) if T >= n_seg else None
    for i in range(n_seg):
        if b is not None: s, e = int(b[i]), max(int(b[i + 1]), int(b[i]) + 1)
        else:             s = min(i * T // n_seg, T - 1); e = s + 1
        x = np.asarray(a[s:e], dtype=np.float32)
        if normalize: x = x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-6)
        out[i] = x.mean(0)
    return np.nan_to_num(out), T

def cache_path(key):
    return CACHE_DIR / (hashlib.md5(key.encode()).hexdigest()[:20] + ".npz")

def process_video(key, files, d_hint):
    cp = cache_path(key)
    if cp.exists():
        return key, "cached"
    pooled, Ts = [], []
    for f in files:
        for view in to_td_views(load_raw(f), d_hint):
            x, T = pool_td(view, N_SEGMENTS, NORMALIZE_FRAMES); pooled.append(x); Ts.append(T)
    x = np.mean(pooled, axis=0).astype(np.float16)               # average over crops (usually just one)
    np.savez(cp.with_suffix(".tmp.npz"), x=x, T=int(Ts[0]))
    os.replace(cp.with_suffix(".tmp.npz"), cp)
    return key, "done"

# reuse a pooled cache that was saved as a Kaggle dataset after an earlier run (skips the slow 28 GB pass)
if not any(CACHE_DIR.glob("*.npz")):
    for dp, dns, _ in os.walk(INPUT_ROOT):
        if CACHE_DIR.name in dns:
            import shutil
            src = Path(dp) / CACHE_DIR.name
            print("re-using pooled cache from", src)
            for f in src.glob("*.npz"): shutil.copy2(f, CACHE_DIR / f.name)
            break

_probe = load_raw(vids.files.iloc[0][0])
EXPECTED_D = int(_probe.shape[-1]) if _probe.ndim == 2 else 512
print("feature dimension:", EXPECTED_D, "| example raw shape:", tuple(_probe.shape))

t0 = time.time()
todo = [(r.key, r.files) for r in vids.itertuples() if not cache_path(r.key).exists()]
print(f"{len(vids) - len(todo)} videos already cached, pooling {len(todo)} ...")
failed = []
if todo:
    def _safe(k, f):
        try: return process_video(k, f, EXPECTED_D)
        except Exception as e: return (k, f"FAILED: {e!r}")
    results = Parallel(n_jobs=IO_THREADS, backend="threading")(delayed(_safe)(k, f) for k, f in tqdm(todo, desc="pooling"))
    failed = [r for r in results if r[1].startswith("FAILED")]
print(f"pooling finished in {(time.time()-t0)/60:.1f} min | failed: {len(failed)}")
for k, m in failed[:5]: print("  ", k, m)

keep, Xs, Ts = [], [], []
for i, r in enumerate(vids.itertuples()):
    cp = cache_path(r.key)
    if cp.exists():
        z = np.load(cp); Xs.append(z["x"]); Ts.append(int(z["T"])); keep.append(i)
vids = vids.iloc[keep].reset_index(drop=True)
X_all = np.stack(Xs).astype(np.float32)                           # [V, N_SEGMENTS, D]
vids["n_frames"] = Ts
print("X_all:", X_all.shape, f"({X_all.nbytes/1e6:.0f} MB) | frames per video: median {int(np.median(Ts))}, max {max(Ts)}")


## Step 6 — Labels and temporal ground truth

In [ ]:
# ---- temporal ground truth (test videos only) ----
def parse_temporal(path):
    ann = {}
    for line in Path(path).read_text(errors="ignore").splitlines():
        parts = line.strip().split()
        if len(parts) < 4: continue
        stems = re.findall(r"[A-Za-z0-9_\-]+x264", parts[0])
        if not stems: continue
        nums = []
        for tok in parts[1:]:
            try: nums.append(int(float(tok)))
            except ValueError: pass
        for s, e in zip(nums[0::2], nums[1::2]):
            if s >= 0 and e > s: ann.setdefault(stems[0].lower(), []).append((s, e))
    return ann

annotations = {}
for p in temporal_files: annotations.update(parse_temporal(p))
vids["has_ann"] = vids.stem.str.lower().isin(annotations)

te = vids[vids.split == "test"]
n_te_anom = int((te.label_bin == 1).sum()); n_ann = int(((te.label_bin == 1) & te.has_ann).sum())
print(f"test anomaly videos with temporal annotation: {n_ann}/{n_te_anom}")

# sanity: annotated frame numbers must fit inside the feature sequence (else FRAME_STRIDE is wrong)
ratios = []
for r in te[te.has_ann].itertuples():
    ratios.append(max(e for _, e in annotations[r.stem.lower()]) / (r.n_frames * FRAME_STRIDE))
if ratios:
    print(f"annotation end-frame / feature length: median {np.median(ratios):.2f}, 95th pct {np.percentile(ratios, 95):.2f}  (should be <= 1)")
    if np.percentile(ratios, 95) > 1.05:
        print("[WARNING] annotations extend beyond the feature sequence -> the features are probably sub-sampled. "
              "Set FRAME_STRIDE (original frames per feature row) in the config cell and re-run from there.")

def frame_gt(stem, n_frames_feat):
    g = np.zeros(n_frames_feat, np.uint8)
    for s, e in annotations.get(stem.lower(), []):
        s, e = int(s // FRAME_STRIDE), int(e // FRAME_STRIDE)
        g[max(s, 0): min(e + 1, n_frames_feat)] = 1
    return g

# ---- index arrays used everywhere below ----
CLS14 = ["Normal"] + CATS
y_bin_all = vids.label_bin.values
y14_all = vids.category.map({c: i for i, c in enumerate(CLS14)}).values
y13_all = np.where(y14_all > 0, y14_all - 1, -1)                   # -1 for Normal
tr_idx_pool = np.where(vids.split.values == "train")[0]
te_idx = np.where(vids.split.values == "test")[0]
print(f"\ntrain pool: {len(tr_idx_pool)} videos | test: {len(te_idx)} videos")


## Step 7 — Quick data check

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
ct = pd.crosstab(vids.category, vids.split).reindex(CLS14)
ct.plot.bar(ax=axes[0], title="Videos per class and split"); axes[0].set_xlabel("")
axes[1].hist(np.log10(vids.n_frames), bins=40); axes[1].set_title("log10(frames per video)")
Xn = X_all[:, :, :]; norms = np.linalg.norm(Xn, axis=2).ravel()
axes[2].hist(norms, bins=60); axes[2].set_title("norm of pooled segment features")
plt.tight_layout(); plt.show()
print(f"binary balance  train pool: Normal {int((y_bin_all[tr_idx_pool]==0).sum())} / Anomaly {int((y_bin_all[tr_idx_pool]==1).sum())}"
      f" | test: Normal {int((y_bin_all[te_idx]==0).sum())} / Anomaly {int((y_bin_all[te_idx]==1).sum())}")


## Step 8 — Metrics and pre-processing helpers

In [ ]:
# =====================  metrics  =====================
def best_threshold(y, p):
    """Threshold maximising F1 of the positive (Anomaly) class -- chosen on OUT-OF-FOLD predictions only."""
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    return float(thr[int(np.argmax(f1))])

def binary_report(y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p), "threshold": thr,
            "accuracy": accuracy_score(y, pred), "balanced_acc": balanced_accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred, zero_division=0),
            "specificity": tn / max(tn + fp, 1), "f1": f1_score(y, pred, zero_division=0),
            "mcc": matthews_corrcoef(y, pred), "brier": brier_score_loss(y, p), "tn": tn, "fp": fp, "fn": fn, "tp": tp}

def topk_acc(y, P, k=3):
    return float(np.mean([y[i] in np.argsort(-P[i])[:k] for i in range(len(y))]))

def multiclass_report(y, P):
    pred = P.argmax(1)
    return {"accuracy": accuracy_score(y, pred), "balanced_acc": balanced_accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y, pred, average="weighted", zero_division=0),
            "top3_acc": topk_acc(y, P, 3), "log_loss": log_loss(y, np.clip(P, 1e-7, 1), labels=list(range(P.shape[1])))}

# =====================  video-level descriptors for the classical models  =====================
def video_descriptor(X):
    """[V, N, D] -> [V, 3D]: mean / max / std over time of the L2-normalised segment features."""
    Xn = X / np.maximum(np.linalg.norm(X, axis=2, keepdims=True), 1e-6)
    return np.concatenate([Xn.mean(1), Xn.max(1), Xn.std(1)], axis=1).astype(np.float32)

# =====================  pre-processing for the neural models  =====================
class SegPreproc:
    """L2-normalise -> subtract the NORMAL prototype (AnomalyCLIP-style re-centering) -> per-dim standardise.
    Statistics come from the training videos of the current fold only (no leakage)."""
    def __init__(self, recenter=True):
        self.recenter = recenter
    @staticmethod
    def _l2(X): return X / np.maximum(np.linalg.norm(X, axis=-1, keepdims=True), 1e-6)
    def fit(self, X_fit, X_normal=None):
        Xn = self._l2(X_fit)
        src = self._l2(X_normal) if (X_normal is not None and len(X_normal)) else Xn
        self.center = src.reshape(-1, src.shape[-1]).mean(0) if self.recenter else Xn.reshape(-1, Xn.shape[-1]).mean(0)
        self.scale = (Xn - self.center).reshape(-1, Xn.shape[-1]).std(0) + 1e-6
        return self
    def transform(self, X):
        return ((self._l2(X) - self.center) / self.scale).astype(np.float32)

# =====================  frame-level evaluation helpers (used for Sultani / AnomalyCLIP-comparable numbers)  =====================
def frames_from_segments(seg_scores, n_seg, T):
    """Repeat each segment score over the frames it covers -> one score per frame."""
    return seg_scores[np.minimum((np.arange(T) * n_seg) // T, n_seg - 1)]

def frame_level_eval(seg_scores, n_seg, video_weight=None, thr=0.5):
    """seg_scores: [n_test, n_seg] aligned with te_idx (sigmoid scores in [0,1]).
       Returns the standard frame-level AUC (all test frames pooled), AP, and Sultani's false-alarm rate on NORMAL test videos
       (fraction of normal-video frames scored >= thr). Anomalous videos without temporal annotation are skipped."""
    G, S, pv_auc, skipped = [], [], [], 0
    far_fr = far_tot = far_vid = n_norm = 0
    for i, g in enumerate(te_idx):
        r = vids.iloc[g]; T = int(r.n_frames)
        if r.label_bin == 1 and not r.has_ann: skipped += 1; continue
        gt = frame_gt(r.stem, T) if r.label_bin == 1 else np.zeros(T, np.uint8)
        s = frames_from_segments(seg_scores[i], n_seg, T)
        if video_weight is not None: s = s * video_weight[i]
        G.append(gt); S.append(s)
        if r.label_bin == 0:
            far_fr += int((s >= thr).sum()); far_tot += T; far_vid += int((s >= thr).any()); n_norm += 1
        elif 0 < gt.sum() < T:
            pv_auc.append(roc_auc_score(gt, s))
    Gc, Sc = np.concatenate(G), np.concatenate(S)
    return {"auc": roc_auc_score(Gc, Sc), "ap": average_precision_score(Gc, Sc),
            "far_frame": far_fr / max(far_tot, 1), "far_video": far_vid / max(n_norm, 1),
            "mean_per_video_auc": float(np.mean(pv_auc)) if pv_auc else float("nan"),
            "n_videos": len(G), "skipped_no_ann": skipped, "anomalous_frame_rate": float(Gc.mean())}

def frame_level_mauc(seg_scores, n_seg, class_probs, video_weight=None):
    """AnomalyCLIP-style VAR metric: for each anomaly class, AUC with that class's annotated anomalous frames as positives and
       ALL other frames of the (annotated-anomaly + normal) test videos as negatives; mean over classes.
       Class score of a frame = anomaly score(frame) * P(class | video).   [approximation: see notebook text]"""
    S_l, W_l, L_l = [], [], []
    for i, g in enumerate(te_idx):
        r = vids.iloc[g]; T = int(r.n_frames)
        if r.label_bin == 1 and not r.has_ann: continue
        lab = np.full(T, -1, np.int16)
        if r.label_bin == 1: lab[frame_gt(r.stem, T) == 1] = y13_all[g]
        s = frames_from_segments(seg_scores[i], n_seg, T)
        if video_weight is not None: s = s * video_weight[i]
        S_l.append(s); W_l.append(class_probs[i]); L_l.append(lab)
    L = np.concatenate(L_l); aucs = {}
    for c in range(13):
        pos = L == c
        if pos.sum() == 0 or pos.all(): aucs[CATS[c]] = float("nan"); continue
        aucs[CATS[c]] = roc_auc_score(pos, np.concatenate([s * w[c] for s, w in zip(S_l, W_l)]))
    return aucs, float(np.nanmean(list(aucs.values())))


## Step 9 — Temporal neural model (MIL for binary, attention pooling for 13-class)

In [ ]:
# =====================  temporal neural model  =====================
class SegNet(nn.Module):
    """Segment sequence [B, N, D] -> temporal encoder -> head.
       task='binary': one anomaly logit per segment; video logit = mean of the top-k segment logits (MIL).
       task='multi' : gated-attention pooling over segments -> class logits."""
    def __init__(self, in_dim, n_seg, hidden, n_layers, n_heads, dropout, in_dropout, arch, task, n_classes=1, topk_frac=0.1):
        super().__init__()
        self.task, self.arch, self.n_seg = task, arch, n_seg
        self.k = max(1, int(round(topk_frac * n_seg)))
        self.inp = nn.Sequential(nn.LayerNorm(in_dim), nn.Dropout(in_dropout), nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(dropout))
        if arch == "transformer":
            self.pos = nn.Parameter(torch.zeros(1, n_seg, hidden)); nn.init.trunc_normal_(self.pos, std=0.02)
            layer = nn.TransformerEncoderLayer(hidden, n_heads, dim_feedforward=hidden * 2, dropout=dropout,
                                               batch_first=True, norm_first=True, activation="gelu")
            self.enc = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        elif arch == "bigru":
            self.enc = nn.GRU(hidden, hidden // 2, num_layers=n_layers, batch_first=True, bidirectional=True,
                              dropout=dropout if n_layers > 1 else 0.0)
        else:  # conv
            blocks = []
            for _ in range(n_layers):
                blocks += [nn.Conv1d(hidden, hidden, 3, padding=1), nn.GELU(), nn.Dropout(dropout)]
            self.enc = nn.Sequential(*blocks)
        self.norm = nn.LayerNorm(hidden)
        if task == "binary":
            self.head = nn.Linear(hidden, 1)
        else:
            self.att_v = nn.Linear(hidden, hidden // 2); self.att_u = nn.Linear(hidden, hidden // 2)
            self.att_w = nn.Linear(hidden // 2, 1); self.cls = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, n_classes))

    def encode(self, x):
        h = self.inp(x)
        if self.arch == "transformer": h = self.enc(h + self.pos)
        elif self.arch == "bigru":     h = self.enc(h)[0]
        else:                          h = h + self.enc(h.transpose(1, 2)).transpose(1, 2)
        return self.norm(h)

    def forward(self, x):
        h = self.encode(x)
        if self.task == "binary":
            seg = self.head(h).squeeze(-1)                                     # [B, N]
            vid = seg.topk(self.k, dim=1).values.mean(1)                       # [B]
            return vid, seg
        a = self.att_w(torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))).squeeze(-1)
        w = torch.softmax(a, dim=1)                                            # [B, N]
        return self.cls((w.unsqueeze(-1) * h).sum(1)), w

def build_optimizer(p, model):
    name = p["optimizer"]
    if name == "adamw": return torch.optim.AdamW(model.parameters(), lr=p["lr"], weight_decay=p["wd"], betas=(0.9, p["beta2"]))
    if name == "radam": return torch.optim.RAdam(model.parameters(), lr=p["lr"], weight_decay=p["wd"])
    return torch.optim.SGD(model.parameters(), lr=p["lr"], momentum=0.9, nesterov=True, weight_decay=p["wd"])

def make_scheduler(opt, epochs, warmup_frac=0.08):
    warm = max(1, int(round(warmup_frac * epochs)))
    def f(e): return (e + 1) / warm if e < warm else 0.5 * (1 + math.cos(math.pi * (e - warm) / max(1, epochs - warm)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def make_model(p, task, in_dim, n_classes=1):
    return SegNet(in_dim, N_SEGMENTS, p["hidden"], p["n_layers"], p["n_heads"], p["dropout"], p["in_dropout"],
                  p["arch"], task, n_classes, p.get("topk_frac", 0.1)).to(DEVICE)

@torch.no_grad()
def predict_nn(model, X, task, bs=256):
    model.eval(); outs, extra = [], []
    for i in range(0, len(X), bs):
        xb = torch.as_tensor(X[i:i + bs], device=DEVICE)
        o, e = model(xb)
        outs.append((torch.sigmoid(o) if task == "binary" else torch.softmax(o, 1)).cpu().numpy())
        extra.append((torch.sigmoid(e) if task == "binary" else e).cpu().numpy())
    return np.concatenate(outs), np.concatenate(extra)            # video probs, segment scores (binary) / attention (multi)

def fit_nn(Xtr, ytr, Xva, yva, p, task, seed, n_classes=1, class_weights=None, report=None):
    """Train one network. Early stopping on val ROC-AUC (binary) / val loss (multi). Returns best model + best val score."""
    seed_everything(seed)
    model = make_model(p, task, Xtr.shape[-1], n_classes)
    opt = build_optimizer(p, model); epochs = p["epochs"]; sched = make_scheduler(opt, epochs)
    Xt = torch.as_tensor(Xtr, device=DEVICE); yt = torch.as_tensor(ytr, device=DEVICE)
    cw = None if class_weights is None else torch.as_tensor(class_weights, dtype=torch.float32, device=DEVICE)
    best, best_state, bad, best_ep = -1e9, None, 0, 0
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xt), device=DEVICE)
        for i in range(0, len(Xt), p["batch_size"]):
            idx = perm[i:i + p["batch_size"]]
            if len(idx) < 2: continue
            xb, yb = Xt[idx], yt[idx]
            if p["noise"] > 0: xb = xb + p["noise"] * torch.randn_like(xb)                       # feature noise
            if p["seg_mask"] > 0:                                                                  # random temporal masking
                xb = xb * (torch.rand(xb.shape[0], xb.shape[1], 1, device=DEVICE) > p["seg_mask"]).float()
            o, seg = model(xb)
            if task == "binary":
                ys = yb.float() * (1 - p["label_smooth"]) + 0.5 * p["label_smooth"]
                loss = F.binary_cross_entropy_with_logits(o, ys)
                ps = torch.sigmoid(seg); an = yb == 1
                if an.any(): loss = loss + p["l_sparse"] * ps[an].mean()                           # few anomalous segments
                loss = loss + p["l_smooth"] * ((ps[:, 1:] - ps[:, :-1]) ** 2).mean()               # temporal smoothness
            else:
                loss = F.cross_entropy(o, yb, weight=cw, label_smoothing=p["label_smooth"])
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step()
        if Xva is not None:
            pv, _ = predict_nn(model, Xva, task)
            if task == "binary": score = roc_auc_score(yva, pv)
            else: score = -log_loss(yva, np.clip(pv, 1e-7, 1), labels=list(range(n_classes)))
            if score > best + 1e-5:
                best, bad, best_ep = score, 0, ep
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= PATIENCE: break
    if best_state is not None: model.load_state_dict(best_state)
    return model, best, best_ep


## Step 10 — Hyper-parameter search / cross-validation engine

In [ ]:
# =====================  task definitions  =====================
D_all = video_descriptor(X_all)                                       # [V, 3*D] for LogReg / LightGBM
anom_pool_ids = tr_idx_pool[y_bin_all[tr_idx_pool] == 1]
normal_pool_ids = tr_idx_pool[y_bin_all[tr_idx_pool] == 0]
TASKS = {
    "binary": dict(ids=tr_idx_pool,   y=y_bin_all, strat=y14_all, n_out=1,  n_classes=2,  metric="ROC-AUC"),
    "multi":  dict(ids=anom_pool_ids, y=y13_all,   strat=y13_all, n_out=13, n_classes=13, metric="macro-F1"),
}
def task_metric(task, y, p):
    return roc_auc_score(y, p) if task == "binary" else f1_score(y, p.argmax(1), average="macro", zero_division=0)

def make_folds(ids, strat, n_splits, seed):
    s = strat[ids]; counts = np.bincount(s)
    n = int(max(2, min(n_splits, counts[counts > 0].min())))
    return [(ids[a], ids[b]) for a, b in StratifiedKFold(n, shuffle=True, random_state=seed).split(ids, s)]

def run_study(name, objective, enqueue=None):
    sampler = optuna.samplers.TPESampler(seed=SEED)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=6, n_warmup_steps=0)
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner, study_name=name)
    for e in (enqueue or []): study.enqueue_trial(e)
    t0 = time.time()
    study.optimize(objective, n_trials=N_TRIALS[name], timeout=TIMEOUT_MIN[name] * 60, gc_after_trial=True)
    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    print(f"[{name}] {len(done)} complete / {len(study.trials)} trials in {(time.time()-t0)/60:.1f} min | best CV score = {study.best_value:.4f}")
    return study

# =====================  neural models: search space, CV, final ensemble  =====================
def suggest_nn(trial, task):
    opt = trial.suggest_categorical("optimizer", ["adamw", "radam", "sgd"])
    d = dict(optimizer=opt,
             arch=trial.suggest_categorical("arch", ["transformer", "bigru", "conv"]),
             hidden=trial.suggest_categorical("hidden", [128, 192, 256, 384]),
             n_layers=trial.suggest_int("n_layers", 1, 3),
             n_heads=trial.suggest_categorical("n_heads", [2, 4, 8]),
             dropout=trial.suggest_float("dropout", 0.1, 0.6),
             in_dropout=trial.suggest_float("in_dropout", 0.0, 0.5),
             wd=trial.suggest_float("wd", 1e-6, 1e-1, log=True),
             beta2=trial.suggest_float("beta2", 0.95, 0.999),
             batch_size=trial.suggest_categorical("batch_size", [16, 32, 64]),
             epochs=trial.suggest_categorical("epochs", [30, 45, 60]),
             noise=trial.suggest_float("noise", 0.0, 0.5),
             seg_mask=trial.suggest_float("seg_mask", 0.0, 0.2),
             label_smooth=trial.suggest_float("label_smooth", 0.0, 0.1),
             recenter=trial.suggest_categorical("recenter", [True, False]))
    if opt == "sgd": d["lr_sgd"] = trial.suggest_float("lr_sgd", 1e-3, 1e-1, log=True)
    else:            d["lr"] = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    if task == "binary":
        d["topk_frac"] = trial.suggest_float("topk_frac", 0.03, 0.3)
        d["l_sparse"] = trial.suggest_float("l_sparse", 1e-4, 1e-1, log=True)
        d["l_smooth"] = trial.suggest_float("l_smooth", 1e-3, 1.0, log=True)
    else:
        d["class_weight"] = trial.suggest_categorical("class_weight", ["none", "balanced"])
    return d

def build_nn_params(d, task):
    p = dict(d)
    p["lr"] = d.get("lr", d.get("lr_sgd"))
    p["epochs"] = int(min(d["epochs"], MAX_EPOCHS))
    p.setdefault("beta2", 0.99)
    if task == "binary":
        p.setdefault("topk_frac", 0.1); p.setdefault("l_sparse", 1e-3); p.setdefault("l_smooth", 1e-2)
    return p

DEFAULT_NN = dict(optimizer="adamw", arch="transformer", hidden=256, n_layers=2, n_heads=4, dropout=0.3, in_dropout=0.2,
                  wd=1e-2, beta2=0.99, batch_size=32, epochs=45, noise=0.1, seg_mask=0.05, label_smooth=0.0,
                  recenter=True, lr=5e-4)
DEFAULT_NN_TASK = {"binary": dict(DEFAULT_NN, topk_frac=0.1, l_sparse=1e-3, l_smooth=1e-2),
                   "multi":  dict(DEFAULT_NN, class_weight="balanced")}

def fit_fold_nn(p, task, tr, va, seed):
    cfg = TASKS[task]
    normal_ids = tr[y_bin_all[tr] == 0] if task == "binary" else normal_pool_ids
    pre = SegPreproc(p["recenter"]).fit(X_all[tr], X_all[normal_ids])
    cw = None
    if task == "multi" and p.get("class_weight") == "balanced":
        cnt = np.bincount(cfg["y"][tr], minlength=13).astype(float)
        cw = len(tr) / (13 * np.maximum(cnt, 1))
    model, score, ep = fit_nn(pre.transform(X_all[tr]), cfg["y"][tr], pre.transform(X_all[va]), cfg["y"][va], p, task, seed,
                              n_classes=cfg["n_out"] if task == "multi" else 1, class_weights=cw)
    return model, pre

def nn_objective_factory(task):
    cfg = TASKS[task]; folds = make_folds(cfg["ids"], cfg["strat"], N_FOLDS_TUNE, SEED)
    def objective(trial):
        p = build_nn_params(suggest_nn(trial, task), task); scores = []
        for k, (tr, va) in enumerate(folds):
            model, pre = fit_fold_nn(p, task, tr, va, SEED + k)
            pv, _ = predict_nn(model, pre.transform(X_all[va]), task)
            scores.append(task_metric(task, cfg["y"][va], pv))
            trial.report(float(np.mean(scores)), k)
            if trial.should_prune(): raise optuna.TrialPruned()
        return float(np.mean(scores))
    return objective

NN_BANK = {"binary": [], "multi": []}          # trained networks kept for saving / inference

def final_nn(task, best_d):
    cfg = TASKS[task]; p = build_nn_params(best_d, task); ids = cfg["ids"]; pos = {g: i for i, g in enumerate(ids)}
    folds = make_folds(ids, cfg["strat"], N_FOLDS_FINAL, SEED + 1)
    oof = np.zeros((len(ids),) if task == "binary" else (len(ids), 13)); test_p, test_seg = [], []
    NN_BANK[task] = []
    for k, (tr, va) in enumerate(folds):
        for s in range(N_SEEDS_FINAL):
            model, pre = fit_fold_nn(p, task, tr, va, SEED + 100 * k + s)
            pv, _ = predict_nn(model, pre.transform(X_all[va]), task)
            oof[[pos[g] for g in va]] += pv / N_SEEDS_FINAL
            pt, sg = predict_nn(model, pre.transform(X_all[te_idx]), task)
            test_p.append(pt); test_seg.append(sg)
            NN_BANK[task].append({"state": {k_: v.cpu() for k_, v in model.state_dict().items()},
                                  "center": pre.center, "scale": pre.scale, "params": p})
        print(f"   fold {k+1}/{len(folds)}  OOF {cfg['metric']} so far: "
              f"{task_metric(task, cfg['y'][np.concatenate([f[1] for f in folds[:k+1]])], oof[[pos[g] for g in np.concatenate([f[1] for f in folds[:k+1]])]]):.4f}")
    return {"oof": oof, "test": np.mean(test_p, 0), "test_seg": np.mean(test_seg, 0) if task == "binary" else None, "params": p}

# =====================  classical models (LogReg, LightGBM) on video descriptors  =====================
def suggest_lr(trial, task):
    return {"C": trial.suggest_float("C", 1e-4, 1e2, log=True)}
def suggest_lgbm(trial, task):
    d = dict(num_leaves=trial.suggest_int("num_leaves", 4, 48), max_depth=trial.suggest_int("max_depth", 2, 8),
             learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
             n_estimators=trial.suggest_int("n_estimators", 80, 500),
             subsample=trial.suggest_float("subsample", 0.5, 1.0), subsample_freq=1,
             colsample_bytree=trial.suggest_float("colsample_bytree", 0.05, 0.5),
             min_child_samples=trial.suggest_int("min_child_samples", 3, 40),
             reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 30, log=True),
             reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 3, log=True))
    if task == "multi": d["class_weight"] = trial.suggest_categorical("class_weight", [None, "balanced"])
    return d

def make_classical(kind, d, task, seed=SEED):
    if kind == "lr":
        return make_pipeline(StandardScaler(), LogisticRegression(C=d["C"], max_iter=3000,
                             class_weight="balanced" if task == "multi" else None))
    d = dict(d)
    return lgb.LGBMClassifier(objective="binary" if task == "binary" else "multiclass", random_state=seed,
                              n_jobs=4, verbose=-1, **d)

def cls_proba(est, X, task):
    P = est.predict_proba(X)
    return P[:, 1] if task == "binary" else P

def classical_objective_factory(kind, task):
    cfg = TASKS[task]; folds = make_folds(cfg["ids"], cfg["strat"], N_FOLDS_TUNE, SEED)
    suggest = suggest_lr if kind == "lr" else suggest_lgbm
    def objective(trial):
        d = suggest(trial, task); scores = []
        for k, (tr, va) in enumerate(folds):
            est = make_classical(kind, d, task).fit(D_all[tr], cfg["y"][tr])
            scores.append(task_metric(task, cfg["y"][va], cls_proba(est, D_all[va], task)))
            trial.report(float(np.mean(scores)), k)
            if trial.should_prune(): raise optuna.TrialPruned()
        return float(np.mean(scores))
    return objective

CLASSICAL_BANK = {}
def final_classical(kind, task, best_d):
    cfg = TASKS[task]; ids = cfg["ids"]; pos = {g: i for i, g in enumerate(ids)}
    folds = make_folds(ids, cfg["strat"], N_FOLDS_FINAL, SEED + 1)
    oof = np.zeros((len(ids),) if task == "binary" else (len(ids), 13)); test_p = []; CLASSICAL_BANK[(kind, task)] = []
    for k, (tr, va) in enumerate(folds):
        est = make_classical(kind, best_d, task, SEED + k).fit(D_all[tr], cfg["y"][tr])
        oof[[pos[g] for g in va]] = cls_proba(est, D_all[va], task)
        test_p.append(cls_proba(est, D_all[te_idx], task)); CLASSICAL_BANK[(kind, task)].append(est)
    return {"oof": oof, "test": np.mean(test_p, 0), "params": best_d}

# =====================  one call = tune + final CV for every model family of a task  =====================
def run_task(task):
    out, studies = {}, {}
    pre = "bin" if task == "binary" else "mc"
    for kind, label in [("lr", "LogReg"), ("lgbm", "LightGBM")]:
        print(f"\n--- tuning {label} ({task}) ---")
        studies[label] = run_study(f"{pre}_{kind}", classical_objective_factory(kind, task))
        out[label] = final_classical(kind, task, studies[label].best_trial.params)
    print(f"\n--- tuning temporal neural net ({task}) ---")
    studies["NN"] = run_study(f"{pre}_nn", nn_objective_factory(task), enqueue=[DEFAULT_NN_TASK[task]])
    print("best NN params:", studies["NN"].best_trial.params)
    print("training final NN ensemble (cross-validated, multi-seed) ...")
    out["NN"] = final_nn(task, studies["NN"].best_trial.params)
    return out, studies


# TASK 1 — Binary: Normal vs Anomaly

### 1a. Tune + train (LogReg, LightGBM, temporal NN)

In [ ]:
# ======== TASK 1 : Normal vs Anomaly -- tuning + cross-validated final models (longest cell: ~30-60 min on a Kaggle GPU) ========
seed_everything(SEED)
bin_res, bin_studies = run_task("binary")
print("\nDone. Models trained:", list(bin_res))


### 1b. Evaluation — video level (selection on OOF, test used once)

In [ ]:
cfgB = TASKS["binary"]; y_pool = y_bin_all[cfgB["ids"]]; y_te = y_bin_all[te_idx]

bin_res["Blend (LR+LGBM+NN)"] = {"oof": np.mean([bin_res[m]["oof"] for m in ["LogReg", "LightGBM", "NN"]], 0),
                                 "test": np.mean([bin_res[m]["test"] for m in ["LogReg", "LightGBM", "NN"]], 0)}
rows, thr_of = [], {}
for name, r in bin_res.items():
    thr = best_threshold(y_pool, r["oof"]); thr_of[name] = thr
    mo, mt = binary_report(y_pool, r["oof"], thr), binary_report(y_te, r["test"], thr)
    rows.append({"model": name, "OOF AUC": mo["roc_auc"], "OOF PR-AUC": mo["pr_auc"], "thr (from OOF)": thr,
                 "TEST AUC": mt["roc_auc"], "TEST PR-AUC": mt["pr_auc"], "TEST acc": mt["accuracy"], "TEST bal.acc": mt["balanced_acc"],
                 "TEST prec": mt["precision"], "TEST recall": mt["recall"], "TEST spec.": mt["specificity"],
                 "TEST F1": mt["f1"], "TEST MCC": mt["mcc"]})
bin_table = pd.DataFrame(rows).set_index("model")
pd.set_option("display.width", 250, "display.max_columns", 30)
print("VIDEO-LEVEL  Normal vs Anomaly  (threshold chosen on out-of-fold predictions, test used once):")
print(bin_table.round(4).to_string())

BEST_BIN = bin_table["OOF AUC"].idxmax()                         # selected on OOF only, never on test
print(f"\nSelected binary model (best OOF AUC): {BEST_BIN} | OOF->TEST AUC gap: "
      f"{bin_table.loc[BEST_BIN,'OOF AUC'] - bin_table.loc[BEST_BIN,'TEST AUC']:+.4f}")

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for name, r in bin_res.items():
    f, t, _ = roc_curve(y_te, r["test"]); ax[0].plot(f, t, label=f"{name} ({roc_auc_score(y_te, r['test']):.3f})")
    p, rc, _ = precision_recall_curve(y_te, r["test"]); ax[1].plot(rc, p, label=f"{name} ({average_precision_score(y_te, r['test']):.3f})")
ax[0].plot([0, 1], [0, 1], "k--", alpha=.4); ax[0].set(title="ROC (test, video level)", xlabel="FPR", ylabel="TPR"); ax[0].legend(fontsize=8)
ax[1].set(title="Precision-Recall (test, video level)", xlabel="Recall", ylabel="Precision"); ax[1].legend(fontsize=8)
cm = confusion_matrix(y_te, (bin_res[BEST_BIN]["test"] >= thr_of[BEST_BIN]).astype(int), labels=[0, 1])
ax[2].imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2): ax[2].text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=16, color="white" if cm[i, j] > cm.max() / 2 else "black")
ax[2].set_xticks([0, 1]); ax[2].set_xticklabels(BIN_NAMES); ax[2].set_yticks([0, 1]); ax[2].set_yticklabels(BIN_NAMES)
ax[2].set(title=f"{BEST_BIN}: confusion @ thr={thr_of[BEST_BIN]:.3f}", xlabel="Predicted", ylabel="True")
plt.tight_layout(); plt.show()

# where does the best model fail? error rate per original category on the test set
cat_te = vids.category.values[te_idx]; pred_te = (bin_res[BEST_BIN]["test"] >= thr_of[BEST_BIN]).astype(int)
err = pd.DataFrame({"cat": cat_te, "wrong": pred_te != y_te}).groupby("cat").wrong.agg(["mean", "sum", "count"])
err.columns = ["error_rate", "n_wrong", "n_videos"]; print("\nBinary error rate per original category (test):"); print(err.sort_values("error_rate", ascending=False).round(3).to_string())

# Optuna diagnostics (neural search)
try:
    import optuna.visualization.matplotlib as ovm
    ovm.plot_optimization_history(bin_studies["NN"]); plt.tight_layout(); plt.show()
    ovm.plot_param_importances(bin_studies["NN"]); plt.tight_layout(); plt.show()
except Exception as e:
    print("(optuna plots skipped:", repr(e)[:80], ")")


### 1c. Frame-level evaluation on the test videos (standard UCF-Crime benchmark number + Sultani's false-alarm rate)

In [ ]:
# ---- FRAME-LEVEL anomaly detection on the test set (the usual UCF-Crime benchmark number) ----
# Training used video-level labels only; the temporal annotations of the test videos are used here purely for scoring.
segB = bin_res["NN"]["test_seg"]                                   # [n_test, N] segment anomaly probabilities
vid_prob = bin_res["Blend (LR+LGBM+NN)"]["test"]
fl_nn   = frame_level_eval(segB, N_SEGMENTS)
fl_nn_w = frame_level_eval(segB, N_SEGMENTS, video_weight=vid_prob)
print(f"videos used: {fl_nn['n_videos']} (skipped {fl_nn['skipped_no_ann']} anomaly videos without annotation) | "
      f"anomalous frames: {fl_nn['anomalous_frame_rate']:.2%}")
loc_table = pd.DataFrame([fl_nn, fl_nn_w], index=["NN segment score", "NN segment score x blended video probability"])[
    ["auc", "ap", "far_frame", "far_video", "mean_per_video_auc"]]
loc_table.columns = ["frame AUC", "frame AP", "false-alarm rate @0.5 (frames of normal videos)", "normal videos with any frame >= 0.5", "mean per-video AUC"]
print(loc_table.round(4).to_string())

sel = [g for g in te_idx if vids.label_bin.iloc[g] == 1 and vids.has_ann.iloc[g]][:6]
fig, axes = plt.subplots(max(len(sel), 1), 1, figsize=(13, 2.0 * max(len(sel), 1)))
axes = np.atleast_1d(axes)
for ax, g in zip(axes, sel):
    i = list(te_idx).index(g); T = int(vids.n_frames.iloc[g])
    ax.fill_between(np.arange(T), 0, frame_gt(vids.stem.iloc[g], T), color="red", alpha=.25, step="mid", label="annotated anomaly")
    ax.plot(frames_from_segments(segB[i], N_SEGMENTS, T), color="navy", lw=1.2, label="predicted score"); ax.set_ylim(0, 1)
    ax.set_ylabel(vids.category.iloc[g], fontsize=8)
axes[0].legend(loc="upper right", fontsize=8); plt.tight_layout(); plt.show()


### 1d. Sultani-style baseline on the same CLIP features (32 segments, MIL ranking loss, paper settings) — for a like-for-like comparison with Sultani et al. 2018

In [ ]:
# ---- Sultani-style baseline on the same CLIP features: 32 segments / bag, 512-32-1 FC net, MIL ranking loss ----
# Settings from the paper (CVPR 2018): 3 FC layers (512, 32, 1 units), 60% dropout between layers, ReLU then Sigmoid,
# Adagrad lr 0.001, sparsity = smoothness = 8e-5, mini-batch = 30 positive + 30 negative bags, 32 non-overlapping segments.
# Not tuned. The only addition: the number of iterations is chosen by early stopping on a held-out 15% of the TRAIN videos.
SULT_SEG, SULT_MAX_ITER, SULT_EVAL_EVERY, SULT_PATIENCE = 32, (200 if FAST_DEBUG else 4000), (50 if FAST_DEBUG else 100), 10
SULT_SEEDS = 1 if FAST_DEBUG else 3
assert N_SEGMENTS % SULT_SEG == 0, "N_SEGMENTS must be a multiple of 32 for the Sultani-style baseline"
X32 = X_all.reshape(len(X_all), SULT_SEG, N_SEGMENTS // SULT_SEG, -1).mean(2).astype(np.float32)   # 64 -> 32 segments

class SultaniNet(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 512), nn.ReLU(), nn.Dropout(0.6), nn.Linear(512, 32), nn.Dropout(0.6),
                                 nn.Linear(32, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x).squeeze(-1)                       # [B, 32] anomaly scores in (0, 1)

def mil_ranking_loss(sp, sn, lam1=8e-5, lam2=8e-5):
    rank = F.relu(1 - sp.max(1).values + sn.max(1).values).mean()               # hinge between top-scored instances
    sparse = sp.sum(1).mean()                                                   # few anomalous segments in a positive bag
    smooth = ((sp[:, 1:] - sp[:, :-1]) ** 2).sum(1).mean()                      # temporally smooth scores
    return rank + lam1 * sparse + lam2 * smooth

def train_sultani(seed):
    seed_everything(seed)
    tr, va = train_test_split(tr_idx_pool, test_size=0.15, random_state=seed, stratify=y14_all[tr_idx_pool])
    pre = SegPreproc(recenter=False).fit(X32[tr])                               # L2-normalise + standardise (fit on train only)
    Xt = torch.as_tensor(pre.transform(X32[tr]), device=DEVICE); Xv = torch.as_tensor(pre.transform(X32[va]), device=DEVICE)
    yv = y_bin_all[va]; pos = torch.as_tensor(np.where(y_bin_all[tr] == 1)[0], device=DEVICE); neg = torch.as_tensor(np.where(y_bin_all[tr] == 0)[0], device=DEVICE)
    model = SultaniNet(Xt.shape[-1]).to(DEVICE); opt = torch.optim.Adagrad(model.parameters(), lr=1e-3)
    bag = min(30, len(pos), len(neg)); best, best_state, bad, best_it = -1, None, 0, 0
    for it in range(1, SULT_MAX_ITER + 1):
        model.train()
        ip, ineg = pos[torch.randperm(len(pos), device=DEVICE)[:bag]], neg[torch.randperm(len(neg), device=DEVICE)[:bag]]
        loss = mil_ranking_loss(model(Xt[ip]), model(Xt[ineg]))
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if it % SULT_EVAL_EVERY == 0:
            model.eval()
            with torch.no_grad(): auc = roc_auc_score(yv, model(Xv).max(1).values.cpu().numpy())   # video score = max segment score
            if auc > best + 1e-5: best, bad, best_it = auc, 0, it; best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= SULT_PATIENCE: break
    model.load_state_dict(best_state); model.eval()
    return model, pre, best, best_it

seg_list, vid_list = [], []
for s in range(SULT_SEEDS):
    model, pre, val_auc, best_it = train_sultani(SEED + s)
    with torch.no_grad(): sc = model(torch.as_tensor(pre.transform(X32[te_idx]), device=DEVICE)).cpu().numpy()
    seg_list.append(sc); print(f"  seed {s}: val video-AUC {val_auc:.4f} at iteration {best_it}")
sult_seg = np.mean(seg_list, 0)                                                 # [n_test, 32]
sult_video = sult_seg.max(1)
y_te_b = y_bin_all[te_idx]
sult_fl = frame_level_eval(sult_seg, SULT_SEG)
sultani_res = {"seg": sult_seg, "video_auc": roc_auc_score(y_te_b, sult_video), "video_ap": average_precision_score(y_te_b, sult_video), "frame": sult_fl}
print(f"\nSultani-style baseline on CLIP features (official-protocol metrics, test set, evaluated once):")
print(f"  frame-level AUC            : {sult_fl['auc']:.4f}")
print(f"  frame-level AP             : {sult_fl['ap']:.4f}")
print(f"  false-alarm rate @ 0.5     : {sult_fl['far_frame']:.4f} of frames in normal test videos  ({sult_fl['far_video']:.1%} of normal videos have any such frame)")
print(f"  video-level AUC / AP (max segment score): {sultani_res['video_auc']:.4f} / {sultani_res['video_ap']:.4f}")


# TASK 2 — 13-class anomaly type

### 2a. Tune + train

In [ ]:
# ======== TASK 2 : 13-way anomaly-type classification (anomalous videos only) -- tuning + cross-validated final models ========
seed_everything(SEED)
mc_res, mc_studies = run_task("multi")
print("\nDone. Models trained:", list(mc_res))


### 2b. Evaluation

In [ ]:
cfgM = TASKS["multi"]; y_pool13 = y13_all[cfgM["ids"]]
te_anom = y_bin_all[te_idx] == 1                                    # evaluate type classification on anomalous test videos
y_te13 = y13_all[te_idx][te_anom]

mc_res["Blend (LR+LGBM+NN)"] = {"oof": np.mean([mc_res[m]["oof"] for m in ["LogReg", "LightGBM", "NN"]], 0),
                                "test": np.mean([mc_res[m]["test"] for m in ["LogReg", "LightGBM", "NN"]], 0)}
rows = []
for name, r in mc_res.items():
    mo, mt = multiclass_report(y_pool13, r["oof"]), multiclass_report(y_te13, r["test"][te_anom])
    rows.append({"model": name, "OOF macro-F1": mo["macro_f1"], "OOF acc": mo["accuracy"],
                 "TEST macro-F1": mt["macro_f1"], "TEST acc": mt["accuracy"], "TEST bal.acc": mt["balanced_acc"],
                 "TEST weighted-F1": mt["weighted_f1"], "TEST top-3 acc": mt["top3_acc"], "TEST log-loss": mt["log_loss"]})
mc_table = pd.DataFrame(rows).set_index("model")
print(f"13-CLASS anomaly-type classification | pool: {len(y_pool13)} train anomaly videos | test: {len(y_te13)} anomaly videos")
print(mc_table.round(4).to_string())
BEST_MC = mc_table["OOF macro-F1"].idxmax()
print(f"\nSelected 13-class model (best OOF macro-F1): {BEST_MC}")

P_best = mc_res[BEST_MC]["test"][te_anom]; pred13 = P_best.argmax(1)
print("\nPer-class report on the test set:")
print(classification_report(y_te13, pred13, labels=list(range(13)), target_names=CATS, zero_division=0))

cm = confusion_matrix(y_te13, pred13, labels=list(range(13))); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(1, 2, figsize=(19, 7.5))
im = ax[0].imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(13):
    for j in range(13):
        if cm[i, j]: ax[0].text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cmn[i, j] > .5 else "black")
ax[0].set_xticks(range(13)); ax[0].set_xticklabels(CATS, rotation=60, ha="right"); ax[0].set_yticks(range(13)); ax[0].set_yticklabels(CATS)
ax[0].set(title=f"{BEST_MC}: confusion matrix (counts, colour = row-normalised)", xlabel="Predicted", ylabel="True")
f1c = f1_score(y_te13, pred13, labels=list(range(13)), average=None, zero_division=0)
ax[1].barh(CATS, f1c, color="steelblue"); ax[1].set(title="Per-class F1 (test)", xlim=(0, 1)); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()
try:
    import optuna.visualization.matplotlib as ovm
    ovm.plot_optimization_history(mc_studies["NN"]); plt.tight_layout(); plt.show()
except Exception as e:
    print("(optuna plot skipped:", repr(e)[:80], ")")


# Combined system and artifacts

### Two-stage 14-class result (detector → type classifier)

In [ ]:
# ---- Combined 14-class system:  Normal-vs-Anomaly detector  ->  13-class type classifier ----
p_anom = bin_res[BEST_BIN]["test"]; thr = thr_of[BEST_BIN]
P13_all = mc_res[BEST_MC]["test"]                                   # type probabilities for every test video
pred14 = np.where(p_anom >= thr, 1 + P13_all.argmax(1), 0)
y14_te = y14_all[te_idx]
rep = {"accuracy": accuracy_score(y14_te, pred14), "balanced_acc": balanced_accuracy_score(y14_te, pred14),
       "macro_f1": f1_score(y14_te, pred14, average="macro", zero_division=0),
       "weighted_f1": f1_score(y14_te, pred14, average="weighted", zero_division=0)}
print(f"TWO-STAGE 14-class result on the official test set ({BEST_BIN} -> {BEST_MC}):")
print({k: round(v, 4) for k, v in rep.items()})
present = sorted(set(y14_te) | set(pred14))
print(classification_report(y14_te, pred14, labels=present, target_names=[CLS14[i] for i in present], zero_division=0))


### AnomalyCLIP-style frame-level mAUC (13 anomaly classes)

In [ ]:
# ---- AnomalyCLIP-style recognition metric: frame-level mAUC over the 13 anomaly classes ----
# AnomalyCLIP predicts a class probability for every FRAME. Our 13-class model predicts one distribution per VIDEO, so the frame
# score for class c is approximated as  anomaly_score(frame) * P(class c | video).  This is an approximation of the paper's metric
# (same positives/negatives/averaging, different source of per-frame class scores) -- report it as "mAUC-style", not as identical.
P13_test = mc_res[BEST_MC]["test"]                                              # [n_test, 13] for ALL test videos
segB = bin_res["NN"]["test_seg"]; vid_prob = bin_res["Blend (LR+LGBM+NN)"]["test"]
mauc = {}
mauc["NN segments x P(class)"]                     = frame_level_mauc(segB, N_SEGMENTS, P13_test)
mauc["NN segments x video prob x P(class)"]        = frame_level_mauc(segB, N_SEGMENTS, P13_test, video_weight=vid_prob)
mauc["Sultani-style segments x P(class)"]          = frame_level_mauc(sultani_res["seg"], SULT_SEG, P13_test)
mauc_table = pd.DataFrame({k: v[0] for k, v in mauc.items()}).T
mauc_table["mAUC"] = [v[1] for v in mauc.values()]
print("Per-class frame-level AUC (one-vs-rest) and their mean (mAUC), test set:")
print((mauc_table * 100).round(1).to_string())


### Comparison with published results (checks the official split first)

In [ ]:
# ---- Is this run comparable to the published numbers? Check the split first. ----
n_te_norm = int((y_bin_all[te_idx] == 0).sum()); n_te_anom = int((y_bin_all[te_idx] == 1).sum())
n_te_ann = int(((y_bin_all[te_idx] == 1) & vids.has_ann.values[te_idx]).sum())
n_tr_norm = int((y_bin_all[tr_idx_pool] == 0).sum()); n_tr_anom = int((y_bin_all[tr_idx_pool] == 1).sum())
OFFICIAL_SPLIT_OK = (len(te_idx) == 290 and n_te_anom == 140 and n_te_norm == 150 and n_te_ann == 140
                     and len(tr_idx_pool) == 1610 and n_tr_norm == 800 and n_tr_anom == 810)
print(f"train: {len(tr_idx_pool)} videos ({n_tr_norm} normal + {n_tr_anom} anomalous) | "
      f"test: {len(te_idx)} videos ({n_te_norm} normal + {n_te_anom} anomalous, {n_te_ann} with temporal annotation)")
print("official UCF-Crime: train 1,610 (800 + 810) | test 290 (150 + 140, all 140 annotated)")
if OFFICIAL_SPLIT_OK:
    print(">>> OFFICIAL SPLIT CONFIRMED: frame-level numbers below are comparable with published UCF-Crime results.")
else:
    print(">>> NOT the official split (or some videos are missing): do NOT compare these numbers with the papers. Fix Step 4 / Step 5 first.")

# ---- fill this in from AnomalyCLIP's Table 2 (secondary sources quote different values, so copy it from the paper itself) ----
ANOMALYCLIP_PUBLISHED = {"AUC": None, "mAUC": None}

pub = [  # (method, features, frame AUC %, mAUC %, false-alarm %, source)
    ("Sultani et al. 2018",              "C3D",            75.41, None, 1.9,  "CVPR 2018 (AUC as quoted in later tables)"),
    ("Sultani et al. 2018 (re-run, I3D)", "I3D",            77.92, None, None, "quoted in later comparison tables"),
    ("RTFM",                              "I3D RGB",        84.30, None, None, "ICCV 2021"),
    ("AnomalyCLIP",                       "CLIP ViT-B/16",  ANOMALYCLIP_PUBLISHED["AUC"], ANOMALYCLIP_PUBLISHED["mAUC"], None, "CVIU 2024, Table 2 -> fill in above"),
]
rows = [{"method": m, "features": f, "frame AUC %": a, "mAUC %": u, "false alarm % @0.5": fa, "source": s} for m, f, a, u, fa, s in pub]
def ours(name, fl, m=None):
    return {"method": name, "features": "CLIP ViT-B/16", "frame AUC %": round(100 * fl["auc"], 2),
            "mAUC %": None if m is None else round(100 * m, 2), "false alarm % @0.5": round(100 * fl["far_frame"], 2),
            "source": "this notebook" + ("" if OFFICIAL_SPLIT_OK else " (NON-official split)")}
rows += [ours("OURS: Sultani-style (32 seg, ranking loss)", sultani_res["frame"], mauc["Sultani-style segments x P(class)"][1]),
         ours("OURS: temporal NN (tuned)",                   fl_nn,   mauc["NN segments x P(class)"][1]),
         ours("OURS: temporal NN x blended video prob",      fl_nn_w, mauc["NN segments x video prob x P(class)"][1])]
comparison = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_colwidth", 60)
print("\n" + comparison.to_string(index=False))
print("""
How to read this table
 * Only the frame AUC (and Sultani's false-alarm rate) follow the published protocols exactly. Feature extractors differ between rows:
   the fairest comparisons are 'OURS: Sultani-style' vs 'OURS: temporal NN' (same features, different method) and AnomalyCLIP (same CLIP features).
 * mAUC here is an approximation (frame score for class c = anomaly score x P(class | video)); the paper uses per-frame class probabilities.
 * We pool every video into 32/64 segments once; AnomalyCLIP samples frames randomly during training. Expect differences from that too.""")


### Save results and models

In [ ]:
import joblib
artifacts = OUT / "artifacts"; artifacts.mkdir(exist_ok=True)

def clean(o):
    if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, np.ndarray): return o.tolist()
    return o

summary = {"config": {"N_SEGMENTS": N_SEGMENTS, "NORMALIZE_FRAMES": NORMALIZE_FRAMES, "FRAME_STRIDE": FRAME_STRIDE, "classes_14": CLS14,
                      "best_binary": BEST_BIN, "binary_threshold": thr_of[BEST_BIN], "best_13class": BEST_MC},
           "binary_table": bin_table.round(5).reset_index().to_dict("records"),
           "multiclass_table": mc_table.round(5).reset_index().to_dict("records"),
           "frame_level": loc_table.round(5).reset_index().to_dict("records"),
           "two_stage_14class": rep,
           "official_split_confirmed": bool(OFFICIAL_SPLIT_OK),
           "sultani_style": {k: v for k, v in sultani_res["frame"].items()} | {"video_auc": sultani_res["video_auc"], "video_ap": sultani_res["video_ap"]},
           "mauc_style": {k: {"mAUC": v[1], "per_class": v[0]} for k, v in mauc.items()},
           "comparison_table": comparison.to_dict("records"),
           "best_params": {"binary": {k: bin_studies[k].best_trial.params for k in bin_studies},
                           "multiclass": {k: mc_studies[k].best_trial.params for k in mc_studies}}}
json.dump(clean(summary), open(artifacts / "results.json", "w"), indent=2)

pred_df = pd.DataFrame({"video": vids.key.values[te_idx], "true_category": vids.category.values[te_idx],
                        "p_anomaly_" + BEST_BIN.split()[0]: bin_res[BEST_BIN]["test"], "pred_binary": (p_anom >= thr).astype(int),
                        "pred_14class": [CLS14[i] for i in pred14]})
for m in ["LogReg", "LightGBM", "NN"]: pred_df["p_anomaly_" + m] = bin_res[m]["test"]
for j, c in enumerate(CATS): pred_df["p_" + c] = P13_all[:, j]
pred_df.to_csv(artifacts / "test_predictions.csv", index=False)
comparison.to_csv(artifacts / "comparison_with_papers.csv", index=False); mauc_table.round(5).to_csv(artifacts / "mauc_per_class.csv")
bin_table.round(5).to_csv(artifacts / "binary_results.csv"); mc_table.round(5).to_csv(artifacts / "multiclass_results.csv")

torch.save(NN_BANK, artifacts / "nn_models.pt")                      # state dicts + preprocessing stats + hyper-parameters
joblib.dump(CLASSICAL_BANK, artifacts / "classical_models.joblib")
print("saved to", artifacts); print(sorted(p.name for p in artifacts.iterdir()))
print(f"\nTip: 'Save Version' -> the folder {CACHE_DIR.name} (pooled features) can be re-attached as a dataset to skip the 28 GB pass next time.")


### Inference helper — predict from a raw `T × 512` feature file

In [ ]:
# ---- inference helper: raw per-frame feature file (T x 512 .npy) -> prediction. Doubles as a consistency test. ----
@torch.no_grad()
def _nn_ensemble(task, X1):
    probs, segs = [], []
    for e in NN_BANK[task]:
        p = e["params"]; m = make_model(p, task, X1.shape[-1], 1 if task == "binary" else 13)
        m.load_state_dict({k: v.to(DEVICE) for k, v in e["state"].items()}); m.eval()
        Xp = ((SegPreproc._l2(X1) - e["center"]) / e["scale"]).astype(np.float32)
        pr, ex = predict_nn(m, Xp, task); probs.append(pr); segs.append(ex)
    return np.mean(probs, 0), np.mean(segs, 0)

def predict_video(feature_file):
    views = to_td_views(load_raw(feature_file), EXPECTED_D)
    X1 = np.mean([pool_td(v, N_SEGMENTS, NORMALIZE_FRAMES)[0] for v in views], 0)[None].astype(np.float32)
    D1 = video_descriptor(X1)
    pb = np.mean([_nn_ensemble("binary", X1)[0][0]] +
                 [np.mean([cls_proba(e, D1, "binary")[0] for e in CLASSICAL_BANK[(k, "binary")]]) for k in ("lr", "lgbm")])
    pm = np.mean([_nn_ensemble("multi", X1)[0][0]] +
                 [np.mean([cls_proba(e, D1, "multi")[0] for e in CLASSICAL_BANK[(k, "multi")]], 0) for k in ("lr", "lgbm")], 0)
    return {"p_anomaly": float(pb), "is_anomaly": bool(pb >= thr_of["Blend (LR+LGBM+NN)"]),
            "top3_types": [(CATS[j], round(float(pm[j]), 3)) for j in np.argsort(-pm)[:3]], "segment_scores": _nn_ensemble("binary", X1)[1][0]}

g0 = te_idx[0]; r = vids.iloc[g0]
out = predict_video(vids.files.iloc[g0][0])
print("video:", r.key, "| true:", r.category)
print("prediction:", {k: v for k, v in out.items() if k != "segment_scores"})
print("consistency with the stored test prediction:", round(out["p_anomaly"], 5), "vs", round(float(bin_res["Blend (LR+LGBM+NN)"]["test"][0]), 5))
